## Why we focus on the latest year (e.g., 2024) for the Supplier Fingerprint

The Supplier Summary (“Supplier Fingerprint”) is meant to describe **current dependency**:  
**Where do our imports come from right now, and how concentrated is that dependency?**

We focus on the **latest available year (2024)** because:
- It represents the **most recent trade structure**, making insights more actionable.
- A multi-year supplier fingerprint can become noisy and harder to interpret in a short presentation.
- Our project already covers trends over time using **Yearly Summary** and **Section Summary** (2022–2024).  
  The supplier fingerprint complements those trends by providing a **snapshot of the latest supplier concentration**.

> Note: The approach is still general. We can easily change `YEAR_FOCUS` to any other year, or build fingerprints for multiple years if needed.

---

## What insights we can generate from the Supplier Summary (Supplier Fingerprint)

**1) Top suppliers per section**
- For each section, identify the **Top N countries** supplying imports.
- Insight: “This section relies heavily on a few countries.”

**2) Supplier concentration (Top1 Share / Top5 Share)**
- **Top1 Share**: the import share of the single largest supplier for a section.
- **Top5 Share**: the combined import share of the five largest suppliers for a section.
- Insight: “Dependency is concentrated (high risk) vs diversified (lower risk).”

**3) High-dependency sections**
- Sections where **Top1 Share** exceeds a threshold (e.g., 50%).
- Insight: “These are priority areas for supplier diversification or localization planning.”

**4) Alternatives (#2 and #3 suppliers)**
- For high-dependency sections, identify the **2nd and 3rd suppliers** as near-term alternatives.
- Insight: “If the top supplier is disrupted, these are the closest available substitutes (based on our data).”

---

## Saved files (outputs) and what each one represents

### 1) `supplier_summary_2024.csv`
**What it contains:** a detailed supplier-level table for the focused year.  
**Columns (typical):**
- Year, Section ID, Section, Country
- imports (Million SAR)
- section_imports (total imports of the section)
- share (country share within the section)
- rank (supplier rank within the section)

**Why it’s useful:**  
This is the **base table** for deeper exploration and custom charts (Top suppliers, shares, rank-based filtering).

---

### 2) `top5_suppliers_2024.csv`
**What it contains:** only the Top 5 suppliers per section (rank ≤ 5).  

**Why it’s useful:**  
Quickly supports supplier charts (Top 5 bars) and Top5 concentration metrics without scanning all countries.

---

### 3) `section_supplier_fingerprint_2024.csv`
**What it contains:** a compact, section-level “fingerprint” table (one row per section).  
**Columns (typical):**
- top1_country, top1_share, top1_imports
- top5_share
- country_rank2, share_rank2, imports_rank2
- country_rank3, share_rank3, imports_rank3

**Why it’s useful:**  
This is the **presentation-ready** table: it summarizes dependency + concentration + alternatives in one place, making it ideal for “Decision Cards” and the dashboard view.

---

In [2]:
import pandas as pd

df = pd.read_excel("../data/gastat_foreign_trade_cleaned.xlsx")

YEAR_FOCUS = 2024

imports_df = df[(df["Year"] == YEAR_FOCUS) & (df["Trade Flow"] == "Imports")].copy()
imports_df.head()

,Country ID,Country,Year,Section ID,Section,Trade Flow ID,Trade Flow,Million SAR
42,abw,Aruba,2024,S03,"Animal and vegetable fats, oils, waxex and the...",1,Imports,0.000000
43,abw,Aruba,2024,S13,"Articles of stone, plaster, cement, ceramic, g...",1,Imports,0.000000
44,abw,Aruba,2024,S15,Base metals and their articles,1,Imports,0.142371
45,abw,Aruba,2024,S12,"Footwear, headgear, umbrellas, sticks",1,Imports,0.105548
46,abw,Aruba,2024,S01,Animals and their products,1,Imports,0.000000


In [3]:
supplier_summary = (imports_df
    .groupby(["Year", "Section ID", "Section", "Country"], as_index=False)
    .agg(imports=("Million SAR", "sum"))
)

supplier_summary.head()

,Year,Section ID,Section,Country,imports
0,2024,S01,Animals and their products,Afghanistan,0.005616
1,2024,S01,Animals and their products,Albania,0.000000
2,2024,S01,Animals and their products,Algeria,0.000232
3,2024,S01,Animals and their products,American Samoa,0.000000
4,2024,S01,Animals and their products,Argentina,329.631740


In [4]:
totals = (supplier_summary
          .groupby(["Year","Section ID","Section"], as_index=False)
          .agg(section_imports=("imports","sum")))

supplier_summary = supplier_summary.merge(totals, on=["Year","Section ID","Section"], how="left")
supplier_summary["share"] = supplier_summary["imports"] / supplier_summary["section_imports"]

supplier_summary.head()

,Year,Section ID,Section,Country,imports,section_imports,share
0,2024,S01,Animals and their products,Afghanistan,0.005616,27394.408003,2.050053e-07
1,2024,S01,Animals and their products,Albania,0.000000,27394.408003,0.000000e+00
2,2024,S01,Animals and their products,Algeria,0.000232,27394.408003,8.468882e-09
3,2024,S01,Animals and their products,American Samoa,0.000000,27394.408003,0.000000e+00
4,2024,S01,Animals and their products,Argentina,329.631740,27394.408003,1.203281e-02


In [6]:
supplier_summary["rank"] = supplier_summary.groupby(["Year","Section ID","Section"])["imports"]\
                                          .rank(method="dense", ascending=False)

top5_suppliers = supplier_summary[supplier_summary["rank"] <= 5].copy()

top5_suppliers.sort_values(["Section ID","rank"]).head(30)

,Year,Section ID,Section,Country,imports,section_imports,share,rank
19,2024,S01,Animals and their products,Brazil,4814.400800,27394.408003,0.175744,1.0
137,2024,S01,Animals and their products,Sudan,2758.909323,27394.408003,0.100711,2.0
102,2024,S01,Animals and their products,New Zealand,2451.976886,27394.408003,0.089506,3.0
7,2024,S01,Animals and their products,Australia,1287.277890,27394.408003,0.046991,4.0
132,2024,S01,Animals and their products,Somalia,1285.113605,27394.408003,0.046912,5.0
227,2024,S02,Plant Products,India,6900.910231,34598.626848,0.199456,1.0
314,2024,S02,Plant Products,United States,3001.436726,34598.626848,0.086750,2.0
283,2024,S02,Plant Products,Russian Federation,2060.339004,34598.626848,0.059550,3.0
203,2024,S02,Plant Products,Egypt,1734.194993,34598.626848,0.050123,4.0
180,2024,S02,Plant Products,Brazil,1722.071792,34598.626848,0.049773,5.0


In [7]:
top1 = (supplier_summary
        .sort_values(["Year","Section ID","Section","imports"], ascending=[True, True, True, False])
        .groupby(["Year","Section ID","Section"], as_index=False)
        .first()[["Year","Section ID","Section","Country","share","imports"]]
        .rename(columns={
            "Country":"top1_country",
            "share":"top1_share",
            "imports":"top1_imports"
        }))

top1.head()

,Year,Section ID,Section,top1_country,top1_share,top1_imports
0,2024,S01,Animals and their products,Brazil,0.175744,4814.400800
1,2024,S02,Plant Products,India,0.199456,6900.910231
2,2024,S03,"Animal and vegetable fats, oils, waxex and the...",Indonesia,0.350003,1771.112110
3,2024,S04,"Prepared foodstuffs, beverages and tobacco",Brazil,0.138756,5138.920539
4,2024,S05,Mineral Products,Egypt,0.328194,16810.935959


In [8]:
top5_share = (top5_suppliers
              .groupby(["Year","Section ID","Section"], as_index=False)
              .agg(top5_share=("share","sum")))

top5_share.head()

,Year,Section ID,Section,top5_share
0,2024,S01,Animals and their products,0.459863
1,2024,S02,Plant Products,0.445652
2,2024,S03,"Animal and vegetable fats, oils, waxex and the...",0.711437
3,2024,S04,"Prepared foodstuffs, beverages and tobacco",0.359448
4,2024,S05,Mineral Products,0.752697


In [9]:
alts = (supplier_summary[supplier_summary["rank"].isin([2,3])]
        .pivot_table(index=["Year","Section ID","Section"],
                     columns="rank",
                     values=["Country","share","imports"],
                     aggfunc="first"))

alts.columns = [f"{a}_rank{int(b)}" for a,b in alts.columns]
alts = alts.reset_index()

alts.head()

,Year,Section ID,Section,Country_rank2,Country_rank3,imports_rank2,imports_rank3,share_rank2,share_rank3
0,2024,S01,Animals and their products,Sudan,New Zealand,2758.909323,2451.976886,0.100711,0.089506
1,2024,S02,Plant Products,United States,Russian Federation,3001.436726,2060.339004,0.086750,0.059550
2,2024,S03,"Animal and vegetable fats, oils, waxex and the...",Malaysia,Sultanate of Oman,855.452377,359.432470,0.169053,0.071030
3,2024,S04,"Prepared foodstuffs, beverages and tobacco",United Arab Emirates,Ireland,3188.158513,1898.484624,0.086083,0.051261
4,2024,S05,Mineral Products,United Arab Emirates,Russian Federation,7064.558301,5959.156125,0.137919,0.116338


In [10]:
fingerprint = top1.merge(top5_share, on=["Year","Section ID","Section"], how="left")\
                 .merge(alts, on=["Year","Section ID","Section"], how="left")

fingerprint = fingerprint.sort_values("top1_share", ascending=False)

fingerprint.head(20)

,Year,Section ID,Section,top1_country,top1_share,top1_imports,top5_share,Country_rank2,Country_rank3,imports_rank2,imports_rank3,share_rank2,share_rank3
18,2024,S19,Arms and Ammunition; Parts and Accessories The...,United States,0.601105,4754.724811,0.867374,South Korea,China,1055.607134,443.445167,0.133453,0.056061
7,2024,S08,"Skins, leather and their articles, handbags an...",China,0.589012,1484.212659,0.847611,Italy,France,400.968542,102.366926,0.159125,0.040624
11,2024,S12,"Footwear, headgear, umbrellas, sticks",China,0.538066,2549.408624,0.855455,Vietnam,Italy,636.160119,474.009584,0.134265,0.100042
13,2024,S14,"Precious stones or metals and their articles, ...",Switzerland,0.497212,19103.928189,0.916324,United Arab Emirates,China,12138.360647,1465.816847,0.315922,0.038150
19,2024,S20,Miscellaneous Manufactured Articles,China,0.484590,9398.932951,0.682689,Italy,United Arab Emirates,1329.583615,1024.514524,0.068551,0.052822
10,2024,S11,Textiles and their articles,China,0.462382,10606.575355,0.737860,India,Türkiye,2610.176805,1825.847745,0.113788,0.079596
20,2024,S21,Works of arts and antiqes,China,0.440896,8170.399848,0.918422,Hong Kong,United States,4487.897005,1846.461257,0.242179,0.099640
15,2024,S16,"Machinery and mechanical appliances, electrica...",China,0.408125,85299.231052,0.664919,United States,Germany,22922.429176,11291.861869,0.109675,0.054027
2,2024,S03,"Animal and vegetable fats, oils, waxex and the...",Indonesia,0.350003,1771.112110,0.711437,Malaysia,Sultanate of Oman,855.452377,359.432470,0.169053,0.071030
6,2024,S07,"Plastics, rubber and their articles",China,0.341105,9674.420291,0.615913,United Arab Emirates,United States,2786.162186,2081.807981,0.098236,0.073401


In [12]:
supplier_out = f"../data/supplier_summary_{YEAR_FOCUS}.csv"
fingerprint_out = f"../data/section_supplier_fingerprint_{YEAR_FOCUS}.csv"
top5_out = f"../data/top5_suppliers_{YEAR_FOCUS}.csv"

supplier_summary.to_csv(supplier_out, index=False, encoding="utf-8-sig")
fingerprint.to_csv(fingerprint_out, index=False, encoding="utf-8-sig")
top5_suppliers.to_csv(top5_out, index=False, encoding="utf-8-sig")

print("Saved:", supplier_out)
print("Saved:", fingerprint_out)
print("Saved:", top5_out)

Saved: ../data/supplier_summary_2024.csv
Saved: ../data/section_supplier_fingerprint_2024.csv
Saved: ../data/top5_suppliers_2024.csv


In [13]:
import pandas as pd
import plotly.express as px

YEAR = 2024
supplier = pd.read_csv(f"../data/supplier_summary_{YEAR}.csv")
top5 = pd.read_csv(f"../data/top5_suppliers_{YEAR}.csv")
finger = pd.read_csv(f"../data/section_supplier_fingerprint_{YEAR}.csv")

supplier.head(), top5.head(), finger.head()

(   Year Section ID                     Section         Country     imports  \
 0  2024        S01  Animals and their products     Afghanistan    0.005616   
 1  2024        S01  Animals and their products         Albania    0.000000   
 2  2024        S01  Animals and their products         Algeria    0.000232   
 3  2024        S01  Animals and their products  American Samoa    0.000000   
 4  2024        S01  Animals and their products       Argentina  329.631740   
 
    section_imports         share   rank  
 0     27394.408003  2.050053e-07   93.0  
 1     27394.408003  0.000000e+00  105.0  
 2     27394.408003  8.468882e-09  102.0  
 3     27394.408003  0.000000e+00  105.0  
 4     27394.408003  1.203281e-02   23.0  ,
    Year Section ID                     Section      Country      imports  \
 0  2024        S01  Animals and their products    Australia  1287.277890   
 1  2024        S01  Animals and their products       Brazil  4814.400800   
 2  2024        S01  Animals and t

### Chart: Top 10 Sections by Supplier Dependency (Top1 Share)

This horizontal bar chart highlights the **10 sections with the highest dependency on a single supplier country** in the selected year.

- **X-axis (Top1 Share):** the percentage of a section’s total imports that comes from its **largest supplier country**.  
  (e.g., 60% means one country provides 60% of that section’s imports.)
- **Y-axis (Section ID):** the trade section (S01–S21).
- **Hover details:**  
  - **top1_country:** the country with the largest import share for that section  
  - **top5_share:** the combined import share of the top 5 supplier countries (a concentration indicator)

**How to interpret:**
- Higher bars = stronger reliance on one country → **higher supply chain concentration risk**.
- Sections with very high Top1 Share may require **supplier diversification** or **localization planning**.

In [15]:
top10_dep = (finger.sort_values("top1_share", ascending=False)
             .head(10)
             .copy())

fig = px.bar(
    top10_dep.sort_values("top1_share"),
    x="top1_share",
    y="Section ID",
    orientation="h",
    title=f"Top 10 Sections by Supplier Dependency (Top1 Share) — {YEAR}",
    labels={"top1_share":"Top1 Share", "Section":"Section"},
    hover_data={"top1_country": True, "top5_share":":.2f"}
)
fig.update_xaxes(tickformat=".0%")
fig.show()

### Chart: Top 5 Suppliers for the Most Dependent Section

This bar chart drills down into **one specific section** (the section with the highest Top1 Share) to show **who the top supplier countries are** and how much they contribute.

- **Selected section:**  
  `SECTION_PICK` is automatically chosen as the section with the **highest supplier dependency** (highest Top1 Share).  
  `SECTION_NAME` is the section’s descriptive name.

- **X-axis (Country):** the top 5 supplier countries for this section (imports only).
- **Y-axis (imports):** import value from each country in **Million SAR**.
- **Hover details:**  
  - **share:** each country’s share of the section’s total imports  
  - **rank:** the supplier rank within the section (1 = largest supplier)

**How to interpret:**
- If the first supplier dominates (much larger bar + high share), it confirms **high concentration risk**.
- Suppliers ranked #2 and #3 represent the most realistic **near-term alternatives** if diversification is needed.

In [18]:
SECTION_PICK = top10_dep.iloc[0]["Section ID"]
SECTION_NAME = top10_dep.iloc[0]["Section"]

sec_top5 = (top5[top5["Section ID"] == SECTION_PICK]
            .sort_values("rank")
            .copy())

fig = px.bar(
    sec_top5,
    x="Country",
    y="imports",
    title=f"Top 5 Suppliers — {SECTION_PICK} ({SECTION_NAME}) Imports — {YEAR}",
    labels={"imports":"Million SAR"},
    hover_data={"share":":.2%","rank":True}
)
fig.update_layout(xaxis_tickangle=-25)
fig.show()

sec_top5[["rank","Country","imports","share"]]

,rank,Country,imports,share
94,1.0,United States,4754.724811,0.601105
92,2.0,South Korea,1055.607134,0.133453
91,3.0,China,443.445167,0.056061
90,4.0,Belgium,332.836123,0.042078
93,5.0,United Kingdom,274.294570,0.034677
